# My Content

Je compare deux méthodes pour recommander cinq articles par utilisateur.

In [1]:
import sys
from pathlib import Path

from IPython.utils.capture import capture_output

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import load_data, prepare_catalog, time_periods
from src.evaluate import choose_method, evaluate, evaluation_events
from src.experiment import comparative_examples, save_results
from src.recommender import Recommender
from src.train import fit_models

## 1. Préparer les données

Je retire les doublons, les lignes incomplètes et les clics aux dates incohérentes.

In [2]:
clicks, metadata, embeddings, quality = load_data(ROOT / "data/brut")
periods = time_periods(clicks)

print(f"{len(clicks):,} clics retenus sur {quality['raw_clicks']:,}")
print(f"{quality['outside_24h_session']:,} clics hors session et {quality['before_publication']:,} avant publication")
print(f"{clicks.user_id.nunique():,} utilisateurs et {len(metadata):,} articles dans le catalogue")

2,977,561 clics retenus sur 2,988,181
7,331 clics hors session et 3,289 avant publication
322,758 utilisateurs et 364,047 articles dans le catalogue


## 2. Comprendre les modèles

Le collaboratif utilise les lecteurs en commun et le contenu compare les embeddings des articles aux dernières lectures.

## 3. Comparer sur les mêmes clics

Je choisis le modèle sur la validation, puis je compare les méthodes à la popularité sur 2 000 cas de test communs.

In [3]:
catalog, variance = prepare_catalog(metadata, embeddings, clicks, periods["validation_start"])

names = {"collaborative": "Collaboratif", "content": "Contenu", "popular": "Popularité"}


def print_results(evaluation):
    print(f"{evaluation['cases']:,} cas")
    for method, scores in evaluation["methods"].items():
        print(
            f"{names[method]} — Hit Rate@5 {scores['hit_rate_at_5']:.2%}, "
            f"MRR@5 {scores['mrr_at_5']:.4f}, NDCG@5 {scores['ndcg_at_5']:.4f}, "
            f"{scores['distinct_categories']:.2f} catégories par liste"
        )


events, validation_count = evaluation_events(clicks, periods["validation_start"], periods["test_start"])
with capture_output():
    validation, validation_details = evaluate(clicks, catalog, events)
decision = choose_method(validation, require_new_articles=True)
print("Validation")
print_results(validation)
print("Choix avant le test", names[decision["selected_method"]])

Validation
2,000 cas
Collaboratif — Hit Rate@5 2.10%, MRR@5 0.0118, NDCG@5 0.0140, 4.20 catégories par liste
Contenu — Hit Rate@5 1.50%, MRR@5 0.0071, NDCG@5 0.0090, 2.21 catégories par liste
Popularité — Hit Rate@5 1.05%, MRR@5 0.0026, NDCG@5 0.0045, 3.75 catégories par liste
Choix avant le test Contenu


In [4]:
events, test_count = evaluation_events(clicks, periods["test_start"], periods["end"])
with capture_output():
    test, test_details = evaluate(clicks, catalog, events)

print("Test")
print_results(test)

Test
2,000 cas
Collaboratif — Hit Rate@5 2.00%, MRR@5 0.0106, NDCG@5 0.0130, 4.20 catégories par liste
Contenu — Hit Rate@5 1.25%, MRR@5 0.0063, NDCG@5 0.0078, 2.22 catégories par liste
Popularité — Hit Rate@5 1.60%, MRR@5 0.0040, NDCG@5 0.0069, 4.27 catégories par liste


Je retiens le contenu pour recommander des articles sans clic.

## 4. Tester trois cas

Je compare la diversité, les articles sans clic et les nouveaux utilisateurs.

In [5]:
with capture_output():
    model = fit_models(clicks, catalog)
scenarios = comparative_examples(model, clicks, validation_details)

concentration = scenarios["concentration"]
for method in ("content", "collaborative"):
    categories = concentration[f"{method}_categories"]
    print(f"Utilisateur {concentration['user_id']}, {names[method]} — catégories distinctes {len(set(categories))}/5")

cold = scenarios["new_article"]
if cold:
    print(f"Article {cold['article_id']} sans clic — rang {cold['content_rank']} avec le contenu, absent du top 5 collaboratif")

new_user = scenarios["new_user"]
print("Sans préférence, les deux méthodes utilisent le même repli.")
for method, result in new_user["with_first_preference"].items():
    print(f"Après une préférence, {names[method]} — {5 - result['fallback_count']} suggestions personnalisées sur cinq")

Utilisateur 1956, Contenu — catégories distinctes 1/5
Utilisateur 1956, Collaboratif — catégories distinctes 5/5
Article 202652 sans clic — rang 4 avec le contenu, absent du top 5 collaboratif
Sans préférence, les deux méthodes utilisent le même repli.
Après une préférence, Collaboratif — 0 suggestions personnalisées sur cinq
Après une préférence, Contenu — 5 suggestions personnalisées sur cinq


## 5. Exporter le modèle

Je vérifie que le modèle exporté propose cinq articles publiés et non lus.

In [6]:
output = ROOT / "models"
save_results(
    output, model, quality, periods, variance, validation, test, decision,
    validation_count, test_count, validation_details + test_details, scenarios, sample=False,
)
loaded = Recommender.load(output / "recommender.npz")
for method in ("collaborative", "content"):
    result = loaded.recommend(1956, method)
    assert len(set(result["article_ids"])) == 5
    assert set(result["article_ids"]).isdisjoint(loaded.history(1956))
    assert all(article["published_at"] <= loaded.reference_time for article in result["articles"])
    print(names[method], result["article_ids"])

Collaboratif [39891, 263284, 347438, 70039, 342184]
Contenu [36360, 212641, 156767, 159718, 157025]
